# Aerospace: Flight Test Manoeuvre Extraction

Extract specific manoeuvre windows from flight test data using marker annotations, compute aerodynamic parameters within each window, and compare manoeuvres.

**Context:** Flight test sessions are long recordings containing multiple test manoeuvres (e.g. stall tests, Dutch rolls, phugoids). Markers annotate the start and end of each manoeuvre, allowing engineers to extract and analyse specific windows.

In [ ]:
import sys
sys.path.insert(0, '..')
from sqlrace_helpers import (
    init_sqlrace, load_session, session_summary,
    list_parameters, extract_parameters
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import os
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")

SESSION_GUID = "828f3c72-9573-4a74-9803-20ff993d848a"
# Typical flight test parameters
PARAMS = ["Acceleration:Platform", "Pressure:Inlet", "Temperature:Sensors"]

cs = f"DbEngine=SQLite;Data Source={os.path.join(DATA_DIR, 'flight-test-manoeuvre-extraction.ssn2')};"
sm = init_sqlrace()
client_session, session = load_session(sm, SESSION_GUID, connection_string=cs)
session_summary(session)

## Discover manoeuvre markers

Markers define time windows for each test manoeuvre.

In [ ]:
markers = session.Markers
manoeuvres = []
for i in range(markers.Count):
    m = markers[i]
    manoeuvres.append({
        "Name": str(m.Label),
        "Start (ns)": int(m.StartTimestamp),
        "End (ns)": int(m.EndTimestamp),
        "Duration (s)": (int(m.EndTimestamp) - int(m.StartTimestamp)) / 1e9,
    })

df_manoeuvres = pd.DataFrame(manoeuvres)
if not df_manoeuvres.empty:
    print(f"{len(df_manoeuvres)} manoeuvre marker(s) found")
    display(df_manoeuvres)
else:
    print("No markers found. Falling back to full session.")
    # Create a single window for the whole session
    df_manoeuvres = pd.DataFrame([{
        "Name": "Full session",
        "Start (ns)": int(session.StartTime),
        "End (ns)": int(session.EndTime),
        "Duration (s)": (session.EndTime - session.StartTime) / 1e9,
    }])

## Extract data per manoeuvre

In [ ]:
available = set(list_parameters(session))
extract_params = [p for p in PARAMS if p in available]

manoeuvre_data = {}
for _, row in df_manoeuvres.iterrows():
    name = row["Name"]
    df = extract_parameters(session, extract_params,
                            start_time=row["Start (ns)"],
                            end_time=row["End (ns)"])
    # Convert to relative time
    if not df.empty:
        t0 = df.index[0]
        df.index = (df.index - t0) / 1e9
        df.index.name = "time_s"
    manoeuvre_data[name] = df
    print(f"  {name}: {df.shape[0]} samples x {df.shape[1]} params")

## Per-manoeuvre statistics

In [ ]:
summary_rows = []
for name, df in manoeuvre_data.items():
    if df.empty:
        continue
    row = {"Manoeuvre": name, "Duration (s)": df.index[-1]}
    for col in df.columns:
        row[f"{col} mean"] = df[col].mean()
        row[f"{col} std"] = df[col].std()
        row[f"{col} peak"] = df[col].abs().max()
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index("Manoeuvre")
display(df_summary)

## Plot manoeuvre windows

In [ ]:
for name, df in manoeuvre_data.items():
    if df.empty:
        continue
    n_cols = len(df.columns)
    fig, axes = plt.subplots(n_cols, 1, figsize=(12, 3 * n_cols),
                             sharex=True, squeeze=False)
    for i, col in enumerate(df.columns):
        axes[i, 0].plot(df.index, df[col], linewidth=0.8)
        axes[i, 0].set_ylabel(col)
        axes[i, 0].grid(True, alpha=0.3)

    axes[0, 0].set_title(f"Manoeuvre: {name}")
    axes[-1, 0].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()

In [ ]:
client_session.Dispose()
print("Session closed.")